# Portfolio Analysis: Multi-Asset USD Portfolio

Objective: analyze the risk, return and diversification of a five-ETF portfolio (all quoted in USD), build the efficient frontier and compare the optimized portfolio with a global benchmark.

**Notebook structure**
1. Setup and data download
2. Returns, volatility and correlations
3. Performance metrics (Sharpe, max drawdown, beta/alpha)
4. Optimization and efficient frontier
5. Robustness checks, conclusions and export

## 1. Setup and data download

In [ ]:
!pip install -q yfinance pandas numpy matplotlib scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.4f}".format

### Configuration

The tickers are Yahoo Finance tickers. **Check each one before running** by searching the ISIN on Yahoo Finance and making sure the listing is quoted in **USD** (watch out for lines quoted in GBp, i.e. pence). If a ticker does not work, replace it here and the rest of the notebook does not change.

In [ ]:
# readable name -> Yahoo Finance ticker (verify by ISIN)
TICKERS = {
    "EM IMI (EIMI)":        "EIMI.L",   # iShares Core MSCI EM IMI, IE00BKM4GZ66
    "All-World (VWRA)":     "VWRA.L",   # Vanguard FTSE All-World USD Acc, IE00BK5BQT80
    "Corp IG USD (LQDE)":   "LQDE.L",   # iShares $ Corp Bond UCITS ETF USD Dist, IE0032895942
    "Gold (IGLN)":          "IGLN.L",   # iShares Physical Gold ETC, USD line (IGLN), IE00B4ND3602
    "UST 3-7y (CBU7)":      "CBU7.L",   # iShares $ Treasury Bond 3-7yr UCITS ETF USD Acc, IE00B3VWN393
}

BENCHMARK = {"MSCI ACWI (SSAC)": "SSAC.L"}   # iShares MSCI ACWI UCITS ETF USD Acc, IE00B6R52259
RISK_FREE_TICKER = "^IRX"                    # US 13-week T-bill (annual yield in %)

START = "2019-01-01"   # the common window is set by the asset with the shortest history (VWRA, from mid-2019)

In [ ]:
all_tickers = {**TICKERS, **BENCHMARK}
inv = {v: k for k, v in all_tickers.items()}

raw = yf.download(list(all_tickers.values()), start=START,
                  auto_adjust=True, progress=False)

# with auto_adjust=True the 'Close' column is already adjusted for dividends and splits
prices_all = raw["Close"].rename(columns=inv)[list(all_tickers.keys())]
prices_all.tail()

#### Alternative: load CSV files downloaded manually from Yahoo Finance
If `yf.download` does not work, download one CSV per asset (Historical Data, daily frequency), save them in a `data/` folder named after the ticker (e.g. `data/EIMI.L.csv`) and uncomment the cell below in place of the download cell.

In [ ]:
# series = {}
# for name, t in all_tickers.items():
#     df = pd.read_csv(f"data/{t}.csv", index_col="Date", parse_dates=True)
#     series[name] = df["Adj Close"]
# prices_all = pd.concat(series, axis=1)

### Quality checks

Before computing anything we check the quotation currency, the start date of each series and missing values. The asset whose data starts latest determines the common window.

In [ ]:
# 1) quotation currency (must be USD for all)
for name, t in all_tickers.items():
    try:
        cur = yf.Ticker(t).fast_info["currency"]
    except Exception:
        cur = "n/a"
    print(f"{name:22s} {t:8s} currency: {cur}")

# 2) first available date for each series
first_valid = prices_all.apply(lambda s: s.first_valid_index())
print("\nFirst available date:")
print(first_valid.to_string())

# 3) missing values
print("\nMissing values per series:")
print(prices_all.isna().sum().to_string())

### Cleaning and common window

In [ ]:
prices_all = prices_all.dropna()   # keep only the dates on which ALL assets have a price

assets = list(TICKERS.keys())
bench  = list(BENCHMARK.keys())[0]

prices = prices_all[assets]        # portfolio
bench_prices = prices_all[bench]   # benchmark

print(f"Common window: {prices.index[0].date()} -> {prices.index[-1].date()}")
print(f"Daily observations: {len(prices)}")

In [ ]:
# risk-free rate: average 13-week T-bill yield over the common window (annual, decimal)
irx = yf.download(RISK_FREE_TICKER, start=prices.index[0], end=prices.index[-1],
                  progress=False)["Close"].squeeze()
RF = irx.mean() / 100
print(f"Average annual risk-free rate (13w T-bill): {RF:.2%}")

In [ ]:
# all assets rebased to 100
rebased = prices_all / prices_all.iloc[0] * 100
ax = rebased.plot(figsize=(11, 5))
ax.set_title("Adjusted prices rebased to 100")
ax.set_ylabel("Index (base 100)")
ax.legend(loc="upper left", fontsize=8)
plt.show()

### Next step
Section 2 computes daily returns, annualized mean return and volatility, and the correlation matrix.

> **Note:** `prices`, `bench_prices` and `RF` are the variables used in the following sections.

---
## 2. Returns, volatility and correlations

From the adjusted prices we compute **simple daily returns**, then annualize them with 252 trading days:

- expected annual return: daily mean × 252
- annual volatility: daily standard deviation × √252
- annual covariance: daily covariance × 252

In [ ]:
TRADING_DAYS = 252

rets = prices.pct_change().dropna()            # daily asset returns
bench_rets = bench_prices.pct_change().dropna()

mu  = rets.mean() * TRADING_DAYS               # mean annual return
vol = rets.std() * np.sqrt(TRADING_DAYS)       # annual volatility
cov = rets.cov() * TRADING_DAYS                # annual covariance matrix
corr = rets.corr()                             # correlation matrix

summary = pd.DataFrame({
    "Annual return": mu,
    "Annual volatility": vol,
    "Sharpe": (mu - RF) / vol,
})
summary.loc[bench] = [
    bench_rets.mean() * TRADING_DAYS,
    bench_rets.std() * np.sqrt(TRADING_DAYS),
    (bench_rets.mean() * TRADING_DAYS - RF) / (bench_rets.std() * np.sqrt(TRADING_DAYS)),
]
summary.style.format({"Annual return": "{:.2%}", "Annual volatility": "{:.2%}", "Sharpe": "{:.2f}"})

### Correlation matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="RdYlGn_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Correlation of daily returns")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Risk and return comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(vol, mu, s=80)
for name in assets:
    ax.annotate(name, (vol[name], mu[name]), textcoords="offset points", xytext=(6, 6), fontsize=9)
ax.scatter(summary.loc[bench, "Annual volatility"], summary.loc[bench, "Annual return"],
           marker="*", s=200, color="black", label=bench)
ax.set_xlabel("Annual volatility"); ax.set_ylabel("Annual return")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_title("Risk vs return (annualized)")
ax.legend()
plt.show()

### How to read the results
- **Low or negative correlations** between assets (typically gold and Treasuries versus equities) are what makes diversification useful.
- **EIMI and All-World** should be highly correlated, because All-World already includes emerging markets: worth commenting on in the analysis.
- **The historical mean return is a noisy estimate**, especially over short windows: it is the most delicate input of the optimization (section 4).

> Variables produced and used in later sections: `rets`, `bench_rets`, `mu`, `cov`, `corr`, `RF`.

---
## 3. Performance metrics

We build the **initial portfolio** (hand-picked weights, before any optimization) and measure it with standard metrics. It is a "naive" reference against which the optimization of section 4 is compared. The weights can be replaced with those of a real portfolio.

Assumption: constant weights, i.e. daily rebalancing with no transaction costs.

In [ ]:
# initial weights (must sum to 1): edit freely
w = pd.Series({
    "EM IMI (EIMI)": 0.15,
    "All-World (VWRA)": 0.40,
    "Corp IG USD (LQDE)": 0.025,
    "Gold (IGLN)": 0.225,
    "UST 3-7y (CBU7)": 0.20
})
# example with different weights:
# w = pd.Series({"EM IMI (EIMI)": 0.15, "All-World (VWRA)": 0.35, "Corp IG USD (LQDE)": 0.10,
#                "Gold (IGLN)": 0.10, "UST 3-7y (CBU7)": 0.20})

assert abs(w.sum() - 1) < 1e-9, "Weights must sum to 1"
port_rets = rets @ w                     # daily portfolio return
port_rets.name = "Portfolio"
w.to_frame("Weight").style.format("{:.1%}")

### Portfolio return and risk (formulas)
With weights *w*, expected-return vector μ and covariance matrix Σ: **μₚ = wᵀμ** and **σₚ = √(wᵀΣw)**. The second number shows the benefit of diversification: it is lower than the weighted average of the individual volatilities whenever correlations are below 1.

In [ ]:
mu_p  = w @ mu
vol_p = np.sqrt(w @ cov @ w)
print(f"Expected annual return (wᵀμ): {mu_p:.2%}")
print(f"Annual volatility       (√wᵀΣw): {vol_p:.2%}")
print(f"Weighted average of individual volatilities: {(w * vol).sum():.2%}  -> diversification benefit: {(w * vol).sum() - vol_p:.2%}")

### Metrics table: assets, portfolio and benchmark

In [ ]:
def perf_metrics(r, rf=RF):
    n = len(r)
    cum = (1 + r).cumprod()
    cagr = cum.iloc[-1] ** (TRADING_DAYS / n) - 1          # compound annual growth rate (CAGR)
    vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe = (r.mean() * TRADING_DAYS - rf) / vol
    dd = cum / cum.cummax() - 1                            # drawdown from the previous peak
    var95 = -np.percentile(r, 5)                           # historical 95% one-day VaR
    cvar95 = -r[r <= np.percentile(r, 5)].mean()           # average loss beyond the VaR
    return pd.Series({
        "CAGR": cagr, "Volatility": vol, "Sharpe": sharpe,
        "Max drawdown": dd.min(), "VaR 95% (1d)": var95, "CVaR 95% (1d)": cvar95,
    })

series = {a: rets[a] for a in assets}
series["Portfolio"] = port_rets
series[bench] = bench_rets

metrics = pd.DataFrame({k: perf_metrics(v) for k, v in series.items()}).T
metrics.style.format({
    "CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.2f}",
    "Max drawdown": "{:.2%}", "VaR 95% (1d)": "{:.2%}", "CVaR 95% (1d)": "{:.2%}",
})

### Beta, alpha and relative risk versus the benchmark
- **Beta**: sensitivity of the portfolio to the benchmark (1 = moves like the market).
- **Alpha (Jensen)**: annual return not explained by beta, computed on *excess returns* over the risk-free rate.
- **Tracking error**: annual volatility of the return difference versus the benchmark.
- **Information ratio**: annual excess return per unit of tracking error.

In [ ]:
rf_d = RF / TRADING_DAYS
ex_p = port_rets - rf_d
ex_b = bench_rets - rf_d

beta = ex_p.cov(ex_b) / ex_b.var()
alpha_ann = (ex_p.mean() - beta * ex_b.mean()) * TRADING_DAYS
r2 = ex_p.corr(ex_b) ** 2

active = port_rets - bench_rets
tracking_error = active.std() * np.sqrt(TRADING_DAYS)
info_ratio = active.mean() * TRADING_DAYS / tracking_error

pd.Series({
    "Beta": beta, "Annual alpha": alpha_ann, "R²": r2,
    "Tracking error": tracking_error, "Information ratio": info_ratio,
}).to_frame("Value").style.format({
    "Value": lambda v: f"{v:.2f}"
})

### Cumulative growth and drawdown

In [ ]:
cum_p = (1 + port_rets).cumprod() * 100
cum_b = (1 + bench_rets).cumprod() * 100
dd_p = cum_p / cum_p.cummax() - 1
dd_b = cum_b / cum_b.cummax() - 1

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(cum_p, label="Portfolio")
axes[0].plot(cum_b, label=bench, color="black", alpha=0.7)
axes[0].set_title("Growth of 100 invested"); axes[0].legend()

axes[1].fill_between(dd_p.index, dd_p, 0, alpha=0.4, label="Portfolio")
axes[1].plot(dd_b, color="black", alpha=0.7, label=bench)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[1].set_title("Drawdown"); axes[1].legend()
plt.tight_layout()
plt.show()

### How to read the results
- **Sharpe**: the higher, the better the risk is rewarded. Compare it with the benchmark and the individual assets.
- **Max drawdown**: the largest peak-to-trough loss, useful to see how much the portfolio "hurts" in its worst periods.
- **VaR / CVaR**: with the 95% VaR, the daily loss exceeds that value on the worst 5% of days; CVaR is the average loss on those days.
- **Beta and alpha**: a low beta with positive alpha indicates return not explained by the market, but over a window of a few years they should be read with caution (noisy estimates).
- Compare Treasuries and corporate IG: the latter carry more credit risk and tend to be more correlated with equities in periods of stress.

> Variables used in the next section: `w`, `port_rets`, `metrics`, `mu`, `cov`, `RF`.

---
## 4. Optimization and efficient frontier

We look for the weights that, given μ (expected returns) and Σ (covariance), produce:

- the **minimum-variance** portfolio (lowest possible risk);
- the **maximum-Sharpe** portfolio (tangency portfolio: best return per unit of risk);
- the **efficient frontier**: for each return level, the portfolio with the lowest risk.

**Constraints**: long-only (no short selling), weights summing to 1 and a **maximum weight per asset** (`MAX_W`). Without a cap, the optimizer tends to concentrate everything in 1-2 assets, a theoretically "optimal" but unrealistic solution.

### Data check before optimizing
Optimizers amplify data errors: we check the most extreme daily moves, especially for the most volatile asset.

In [ ]:
extreme = rets.abs().idxmax()
print("Day of the most extreme daily move, per asset:")
print(pd.DataFrame({"date": extreme, "return": [rets.loc[d, a] for a, d in extreme.items()]}).to_string())

most_vol = rets.std().idxmax()
print(f"\nFive largest moves (in absolute value) of the most volatile asset ({most_vol}):")
print(rets[most_vol].reindex(rets[most_vol].abs().nlargest(5).index).to_string())

In [ ]:
from scipy.optimize import minimize

n = len(assets)
MAX_W = 0.40                                   # maximum weight per single asset (editable)
bounds = tuple((0.0, MAX_W) for _ in range(n))
sum_to_one = {"type": "eq", "fun": lambda x: x.sum() - 1}
x0 = np.repeat(1 / n, n)

mu_v, cov_v = mu.values, cov.values
port_ret = lambda x: x @ mu_v
port_vol = lambda x: np.sqrt(x @ cov_v @ x)
neg_sharpe = lambda x: -(port_ret(x) - RF) / port_vol(x)

res_minvar = minimize(port_vol,   x0, method="SLSQP", bounds=bounds, constraints=[sum_to_one])
res_maxsh  = minimize(neg_sharpe, x0, method="SLSQP", bounds=bounds, constraints=[sum_to_one])
assert res_minvar.success and res_maxsh.success, "Optimization did not converge"

w_minvar = pd.Series(res_minvar.x, index=assets)
w_maxsh  = pd.Series(res_maxsh.x,  index=assets)

### Efficient frontier

In [ ]:
# maximum return achievable under the constraints
ret_max = -minimize(lambda x: -port_ret(x), x0, method="SLSQP",
                    bounds=bounds, constraints=[sum_to_one]).fun
targets = np.linspace(port_ret(w_minvar.values), ret_max, 50)

frontier_vol, frontier_ret = [], []
x_prev = w_minvar.values
for t in targets:
    cons = [sum_to_one, {"type": "eq", "fun": lambda x, t=t: port_ret(x) - t}]
    r = minimize(port_vol, x_prev, method="SLSQP", bounds=bounds, constraints=cons)
    if r.success:
        frontier_vol.append(port_vol(r.x)); frontier_ret.append(port_ret(r.x)); x_prev = r.x

# cloud of random portfolios (same constraints)
rng = np.random.default_rng(42)
W = rng.dirichlet(np.ones(n), 60000)
W = W[W.max(axis=1) <= MAX_W]
rand_ret = W @ mu_v
rand_vol = np.sqrt(np.einsum("ij,jk,ik->i", W, cov_v, W))
rand_sharpe = (rand_ret - RF) / rand_vol

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
sc = ax.scatter(rand_vol, rand_ret, c=rand_sharpe, cmap="viridis", s=6, alpha=0.5)
fig.colorbar(sc, ax=ax, label="Sharpe")
ax.plot(frontier_vol, frontier_ret, color="red", lw=2.5, label="Efficient frontier")

ax.scatter(vol, mu, marker="o", s=60, color="white", edgecolor="black", zorder=5)
for a in assets:
    ax.annotate(a, (vol[a], mu[a]), textcoords="offset points", xytext=(6, 4), fontsize=8)

ax.scatter(port_vol(w_minvar.values), port_ret(w_minvar.values), marker="*", s=260, color="gold",
           edgecolor="black", zorder=6, label="Minimum variance")
ax.scatter(port_vol(w_maxsh.values), port_ret(w_maxsh.values), marker="*", s=260, color="red",
           edgecolor="black", zorder=6, label="Maximum Sharpe")
ax.scatter(port_vol(w.values), port_ret(w.values), marker="D", s=90, color="blue",
           edgecolor="black", zorder=6, label="Initial portfolio")
ax.scatter(summary.loc[bench, "Annual volatility"], summary.loc[bench, "Annual return"],
           marker="X", s=110, color="black", zorder=6, label=bench)

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_xlabel("Annual volatility"); ax.set_ylabel("Expected annual return")
ax.set_title(f"Efficient frontier (long-only, max weight {MAX_W:.0%})")
ax.legend(loc="lower right", fontsize=8)
plt.show()

### Weights and metrics: initial vs optimized vs benchmark

In [ ]:
weights = pd.DataFrame({"Initial": w, "Minimum variance": w_minvar, "Maximum Sharpe": w_maxsh})
display(weights.style.format("{:.1%}"))

compare = pd.DataFrame({
    "Initial":        perf_metrics(rets @ w),
    "Minimum variance": perf_metrics(rets @ w_minvar),
    "Maximum Sharpe":  perf_metrics(rets @ w_maxsh),
    bench:             perf_metrics(bench_rets),
}).T
compare.style.format({
    "CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.2f}",
    "Max drawdown": "{:.2%}", "VaR 95% (1d)": "{:.2%}", "CVaR 95% (1d)": "{:.2%}",
})

In [ ]:
cum = pd.DataFrame({
    "Initial": (1 + rets @ w).cumprod() * 100,
    "Minimum variance": (1 + rets @ w_minvar).cumprod() * 100,
    "Maximum Sharpe": (1 + rets @ w_maxsh).cumprod() * 100,
    bench: (1 + bench_rets).cumprod() * 100,
})
ax = cum.plot(figsize=(11, 5))
ax.set_title("Growth of 100 invested (in-sample)"); ax.set_ylabel("Value")
plt.show()

### How to read the results (and the limitations)
- **The minimum-variance portfolio** tends to overweight low-volatility assets (here probably Treasuries): low risk, but also low return if that asset returned little over the period.
- **Maximum Sharpe depends on historical returns**, the least stable estimate. If an asset had an exceptional period (e.g. gold), the optimizer overweights it up to the `MAX_W` cap: this does not mean it will be the best performer in the future.
- **In-sample result**: weights are optimized on the same data used to evaluate them, so performance is by construction better than it would be live. This is the main limitation to state in the analysis.
- Trying different `MAX_W` values (0.30, 0.40, 1.00) shows how unstable the solution is: a good robustness test to discuss in the README.

> Variables produced: `w_minvar`, `w_maxsh`, `weights`, `compare`.

---
## 5. Robustness, conclusions and export

The results of section 4 are **in-sample**: the weights are chosen by looking at the same data on which they are evaluated. Here we run two tests to see how well the conclusions hold:

1. **Out-of-sample test**: optimize on the first years (train) and evaluate, with fixed weights, on the following years (test).
2. **Sensitivity to the maximum weight**: how the maximum-Sharpe weights change as `MAX_W` varies.

In [ ]:
def optimize(mu_v, cov_v, max_w=0.40, objective="sharpe", rf=RF):
    """Long-only weights (sum to 1, at most max_w per asset). objective: 'sharpe' or 'minvar'."""
    k = len(mu_v)
    assert max_w * k >= 1, "max_w too low: no weights can sum to 1"
    vol_f = lambda x: np.sqrt(x @ cov_v @ x)
    f = (lambda x: -((x @ mu_v) - rf) / vol_f(x)) if objective == "sharpe" else vol_f
    res = minimize(f, np.repeat(1 / k, k), method="SLSQP",
                   bounds=tuple((0.0, max_w) for _ in range(k)),
                   constraints=[{"type": "eq", "fun": lambda x: x.sum() - 1}])
    assert res.success, "Optimization did not converge"
    return res.x

### 5.1 Out-of-sample test
Weights are optimized on the **train** period only, then applied to the **test** period, which the optimizer never saw. They are compared with the initial (non-optimized) portfolio, equal weights (1/N) and the benchmark. Change `SPLIT` to try other cut-off dates.

In [ ]:
SPLIT = "2023-12-31"
train = rets.loc[:SPLIT]
test  = rets.loc[pd.Timestamp(SPLIT) + pd.Timedelta(days=1):]
print(f"Train: {train.index[0].date()} -> {train.index[-1].date()} ({len(train)} days)")
print(f"Test : {test.index[0].date()} -> {test.index[-1].date()} ({len(test)} days)")

mu_tr  = train.mean() * TRADING_DAYS
cov_tr = train.cov() * TRADING_DAYS

k = len(assets)
w_oos = pd.DataFrame({
    "Max Sharpe (train)":  optimize(mu_tr.values, cov_tr.values, MAX_W, "sharpe"),
    "Min variance (train)": optimize(mu_tr.values, cov_tr.values, MAX_W, "minvar"),
    "Initial": w.values,
    "1/N": np.repeat(1 / k, k),
}, index=assets)
display(w_oos.style.format("{:.1%}"))

oos = pd.DataFrame({name: perf_metrics(test @ w_oos[name]) for name in w_oos.columns}).T
oos.loc[bench] = perf_metrics(bench_rets.loc[test.index])
oos.style.format({
    "CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.2f}",
    "Max drawdown": "{:.2%}", "VaR 95% (1d)": "{:.2%}", "CVaR 95% (1d)": "{:.2%}",
})

In [ ]:
cum_test = pd.DataFrame({name: (1 + test @ w_oos[name]).cumprod() * 100 for name in w_oos.columns})
cum_test[bench] = (1 + bench_rets.loc[test.index]).cumprod() * 100
ax = cum_test.plot(figsize=(11, 5))
ax.set_title(f"Test period (from {test.index[0].date()}): weights optimized on the train period only")
ax.set_ylabel("Value of 100 invested")
plt.show()

### 5.2 Sensitivity to the maximum weight per asset

In [ ]:
rows = {}
for mw in [0.25, 0.30, 0.40, 0.50, 0.60, 1.00]:
    x = optimize(mu.values, cov.values, mw, "sharpe")
    m = perf_metrics(rets @ x)
    rows[f"max {mw:.0%}"] = pd.concat([pd.Series(x, index=assets),
                                       m[["CAGR", "Volatility", "Sharpe"]]])
sens = pd.DataFrame(rows).T
fmt = {a: "{:.1%}" for a in assets}
fmt.update({"CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.2f}"})
sens.style.format(fmt)

### How to read the tests
- **Out-of-sample**: if the maximum-Sharpe portfolio (train) keeps an edge over the initial portfolio, 1/N and the benchmark on the test period, the optimization added real value; if it loses it, the in-sample result was largely due to the period. Both outcomes should be reported honestly.
- **Sensitivity**: if the weights change a lot as `MAX_W` varies (or if a single asset takes up almost the whole portfolio), the solution is unstable and depends on a few historical returns.

### Exporting the results (tables and charts for the repository)

In [ ]:
import os
os.makedirs("results", exist_ok=True)

weights.to_csv("results/weights.csv")
compare.to_csv("results/metrics_in_sample.csv")
oos.to_csv("results/metrics_out_of_sample.csv")
sens.to_csv("results/sensitivity_max_weight.csv")

# cumulative growth chart (in-sample)
ax = cum.plot(figsize=(11, 5)); ax.set_title("Growth of 100 invested (in-sample)")
ax.figure.savefig("results/growth_of_100.png", dpi=200, bbox_inches="tight"); plt.close(ax.figure)

# efficient frontier
fig, ax = plt.subplots(figsize=(10, 6.5))
sc = ax.scatter(rand_vol, rand_ret, c=rand_sharpe, cmap="viridis", s=6, alpha=0.5)
fig.colorbar(sc, ax=ax, label="Sharpe")
ax.plot(frontier_vol, frontier_ret, color="red", lw=2.5, label="Efficient frontier")
for a in assets:
    ax.scatter(vol[a], mu[a], s=60, color="white", edgecolor="black", zorder=5)
    ax.annotate(a, (vol[a], mu[a]), textcoords="offset points", xytext=(6, 4), fontsize=8)
for wt, mk, col, lab in [(w_minvar, "*", "gold", "Minimum variance"), (w_maxsh, "*", "red", "Maximum Sharpe"),
                         (w, "D", "blue", "Initial portfolio")]:
    ax.scatter(port_vol(wt.values), port_ret(wt.values), marker=mk, s=220, color=col,
               edgecolor="black", zorder=6, label=lab)
ax.set_xlabel("Annual volatility"); ax.set_ylabel("Expected annual return")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_title(f"Efficient frontier (long-only, max weight {MAX_W:.0%})"); ax.legend(loc="lower right", fontsize=8)
fig.savefig("results/efficient_frontier.png", dpi=200, bbox_inches="tight"); plt.close(fig)

print("Saved in results/:", sorted(os.listdir("results")))

## Conclusions and limitations

**What the analysis shows**
- Diversification reduces risk: the portfolio volatility is well below the weighted average of the individual asset volatilities, and the initial portfolio lies below the efficient frontier, so it can be improved at the same level of risk.
- The maximum-Sharpe portfolio beats the benchmark in-sample with lower volatility and a shallower drawdown, but it is concentrated in a few assets (gold and global equity at the cap).
- The minimum-variance portfolio has very low risk, but also a return close to the risk-free rate, hence a low Sharpe ratio.

**Limitations to state**
- In-sample optimization: the weights reflect the historical period (in particular the gold rally and the bond sell-off as rates rose).
- Expected returns are estimated from historical means, which are very unstable; the covariance matrix is more reliable.
- No transaction costs or taxes, idealized daily rebalancing, long-only positions.
- A sample of about seven years covers few market regimes; a constant average risk-free rate simplifies the Sharpe ratio calculation.